# LeetCode 3248 — Snake in Matrix bằng Python

Link đề: https://leetcode.com/problems/snake-in-matrix/ · Độ khó: **Easy** · Chủ đề: Array, String, Simulation (mô phỏng)

Skeleton LeetCode:

```python
class Solution:
    def finalPositionOfSnake(self, n: int, commands: List[str]) -> int:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề (`n = 2` và `n = 3`); những ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc. Mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả đối chiếu hàng nghìn lần với bản mô phỏng vét cạn trên lưới thật).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong một lưới `n x n`, ô ở hàng `i`, cột `j` được **đánh số** `i * n + j`; con rắn xuất phát ở ô `0` rồi đi theo các lệnh `"UP"`, `"RIGHT"`, `"DOWN"`, `"LEFT"`; ta phải trả về **số hiệu của ô cuối cùng**.
2. Nắm các công cụ Python: **danh sách chuỗi** (`commands`), vòng lặp `for`, **chỉ số 2 chiều** `grid[i][j]`, công thức **số ô = hàng × n + cột**, `divmod` để đổi ngược, **`dict` ánh xạ lệnh → độ lệch**, `str.count` / `Counter` và `match`-`case` (Python 3.10+).
3. Đi từ cách **dựng cả lưới và đi trên lưới** → **chỉ theo dõi tọa độ `(x, y)`** → **theo dõi trực tiếp số ô** (`UP = −n`, `DOWN = +n`, `LEFT = −1`, `RIGHT = +1`) → **chỉ cần đếm số lệnh mỗi loại** vì thứ tự không ảnh hưởng đến vị trí cuối.
4. Hiểu mấu chốt: đây là bài **mô phỏng**, nhưng vì mỗi lệnh chỉ **cộng một độ lệch cố định** nên vị trí cuối là **tổng các độ lệch** — không cần dựng lưới, không cần kiểm tra biên (đề đảm bảo rắn không ra ngoài).
5. Tránh các lỗi kinh điển: đảo chiều `UP`/`DOWN`, nhầm công thức `i * n + j` thành `i + j * n`, so sánh `c == "U"` khi lệnh là chuỗi đầy đủ `"UP"`, và nhầm số `n` (độ rộng một hàng) khi nhảy một hàng.

## 2. Đề bài & ví dụ

Có một con rắn trong lưới `grid` kích thước `n x n`, và nó có thể di chuyển theo **bốn hướng**. Mỗi ô trong lưới được xác định bởi vị trí `grid[i][j] = (i * n) + j`.

Con rắn bắt đầu ở ô `0` và làm theo một dãy lệnh.

Cho số nguyên `n` (cỡ lưới) và mảng chuỗi `commands`, mỗi phần tử là một trong `"UP"`, `"RIGHT"`, `"DOWN"`, `"LEFT"`. Đề **đảm bảo** rắn không bao giờ đi ra ngoài lưới.

Hãy trả về **vị trí (số hiệu) của ô cuối cùng** mà con rắn đứng sau khi thực hiện hết `commands`.

**Ví dụ 1**

- Input: `n = 2`, `commands = ["RIGHT", "DOWN"]`
- Output: `3`
- Giải thích: lưới `2 x 2` là `[[0, 1], [2, 3]]`. Rắn ở ô `0` → `RIGHT` sang ô `1` → `DOWN` xuống ô `3`.

**Ví dụ 2**

- Input: `n = 3`, `commands = ["DOWN", "RIGHT", "UP"]`
- Output: `1`
- Giải thích: lưới `3 x 3` là `[[0, 1, 2], [3, 4, 5], [6, 7, 8]]`. Rắn: `0` → `DOWN` ô `3` → `RIGHT` ô `4` → `UP` ô `1`.

**Ràng buộc:** `2 <= n <= 10`; `1 <= commands.length <= 100`; `commands` chỉ gồm `"UP"`, `"RIGHT"`, `"DOWN"`, `"LEFT"`; đề đảm bảo rắn không ra ngoài lưới.

In [1]:
from typing import List
from collections import Counter
import random

examples = [
    (2, ["RIGHT", "DOWN"], 3),
    (3, ["DOWN", "RIGHT", "UP"], 1),
]
for n, commands, expected in examples:
    print(f"n={n}, commands={commands} -> kỳ vọng {expected}")

n=2, commands=['RIGHT', 'DOWN'] -> kỳ vọng 3
n=3, commands=['DOWN', 'RIGHT', 'UP'] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Lưới 2 chiều và công thức `i * n + j`

Một lưới `n x n` có thể lưu bằng **danh sách của các danh sách**: `grid[i]` là hàng `i`, `grid[i][j]` là ô ở hàng `i`, cột `j` (đếm từ `0`). Đề đánh số ô theo kiểu **đọc từ trái sang phải, từ trên xuống dưới**: hàng `0` là `0..n-1`, hàng `1` là `n..2n-1`, ... Vì mỗi hàng có `n` ô nên ô `(i, j)` mang số `i * n + j`.

In [2]:
n = 3
grid = [[i * n + j for j in range(n)] for i in range(n)]
for row in grid:
    print(row)
print("grid[1][2] =", grid[1][2], "và 1*3 + 2 =", 1 * 3 + 2)     # 5 và 5

[0, 1, 2]
[3, 4, 5]
[6, 7, 8]
grid[1][2] = 5 và 1*3 + 2 = 5


### 3.2. Đổi ngược từ số ô về `(hàng, cột)` bằng `divmod`

Từ số ô `p`, hàng là `p // n` (chia nguyên) và cột là `p % n` (phần dư). `divmod(p, n)` trả về cả hai cùng lúc. Đây là phép "đổi hệ" giống như đổi số phút ra (giờ, phút).

In [3]:
n = 3
for p in [0, 4, 5, 8]:
    row, col = divmod(p, n)
    print(f"ô {p} -> hàng {row}, cột {col}  (kiểm tra: {row} * {n} + {col} = {row * n + col})")

ô 0 -> hàng 0, cột 0  (kiểm tra: 0 * 3 + 0 = 0)
ô 4 -> hàng 1, cột 1  (kiểm tra: 1 * 3 + 1 = 4)
ô 5 -> hàng 1, cột 2  (kiểm tra: 1 * 3 + 2 = 5)
ô 8 -> hàng 2, cột 2  (kiểm tra: 2 * 3 + 2 = 8)


### 3.3. Duyệt danh sách chuỗi và lấy ký tự đầu

`commands` là list các chuỗi; `for c in commands` lấy từng lệnh. `c[0]` là **ký tự đầu** của chuỗi (`"UP"[0] == "U"`). Bốn lệnh có **chữ cái đầu khác nhau** (`U`, `R`, `D`, `L`), nên chỉ cần nhìn ký tự đầu là phân biệt được. Nhưng cũng có thể so sánh cả chuỗi: `c == "UP"`.

In [4]:
commands = ["RIGHT", "DOWN", "UP", "LEFT"]
for c in commands:
    print(repr(c), "ký tự đầu:", repr(c[0]), "| c == 'UP'?", c == "UP", "| c == 'U'?", c == "U")

'RIGHT' ký tự đầu: 'R' | c == 'UP'? False | c == 'U'? False
'DOWN' ký tự đầu: 'D' | c == 'UP'? False | c == 'U'? False
'UP' ký tự đầu: 'U' | c == 'UP'? True | c == 'U'? False
'LEFT' ký tự đầu: 'L' | c == 'UP'? False | c == 'U'? False


### 3.4. `dict` ánh xạ lệnh → độ lệch

Thay vì một chuỗi `if`/`elif` dài, ta có thể lưu quy tắc vào **từ điển**: khóa là tên lệnh, giá trị là độ lệch. Ví dụ độ lệch của **tọa độ** `(dx, dy)`: `UP = (-1, 0)` (lên một hàng nên hàng giảm `1`), `DOWN = (1, 0)`, `LEFT = (0, -1)`, `RIGHT = (0, 1)`. Tra cứu `d[c]` mất `O(1)`.

In [5]:
DIRS = {"UP": (-1, 0), "DOWN": (1, 0), "LEFT": (0, -1), "RIGHT": (0, 1)}
dx, dy = DIRS["RIGHT"]
print("RIGHT ->", dx, dy)
x, y = 0, 0
for c in ["RIGHT", "DOWN"]:
    dx, dy = DIRS[c]
    x, y = x + dx, y + dy
    print(f"sau {c}: (x, y) = ({x}, {y})")

RIGHT -> 0 1
sau RIGHT: (x, y) = (0, 1)
sau DOWN: (x, y) = (1, 1)


### 3.5. Đếm phần tử: `list.count` và `Counter`

`commands.count("UP")` cho biết lệnh `"UP"` xuất hiện bao nhiêu lần (quét cả list, `O(m)`). `Counter(commands)` đếm **tất cả** loại trong **một** lượt và cho `cnt["UP"]`; loại không có thì trả `0` (không báo lỗi như `dict` thường).

In [6]:
commands = ["DOWN", "RIGHT", "UP", "DOWN"]
print(commands.count("DOWN"))              # 2
cnt = Counter(commands)
print(cnt)                                 # Counter({'DOWN': 2, 'RIGHT': 1, 'UP': 1})
print(cnt["DOWN"], cnt["LEFT"])            # 2 0   (LEFT không xuất hiện -> 0)

2
Counter({'DOWN': 2, 'RIGHT': 1, 'UP': 1})
2 0


### 3.6. `match` / `case` (Python 3.10+)

`match c:` so khớp giá trị `c` với từng `case`. Nó gọn hơn chuỗi `if / elif`, tương tự `switch` ở ngôn ngữ khác. LeetCode hỗ trợ từ Python 3.10 trở lên. Nếu bạn chưa quen thì dùng `if / elif` hoặc `dict` đều được; kết quả giống hệt.

In [7]:
def huong(c):
    match c:
        case "UP":
            return "lên"
        case "DOWN":
            return "xuống"
        case "LEFT":
            return "trái"
        case _:
            return "phải"

print([huong(c) for c in ["UP", "RIGHT", "DOWN", "LEFT"]])

['lên', 'phải', 'xuống', 'trái']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tự làm Ví dụ 2 trên giấy: `n = 3`, `commands = ["DOWN", "RIGHT", "UP"]`.

```
0 1 2
3 4 5
6 7 8
```

- Bắt đầu ở ô `0` (hàng `0`, cột `0`).
- `DOWN`: xuống một hàng → hàng `1`, cột `0` → ô `3`. Số ô tăng `3` (= `n`).
- `RIGHT`: sang phải một cột → hàng `1`, cột `1` → ô `4`. Số ô tăng `1`.
- `UP`: lên một hàng → hàng `0`, cột `1` → ô `1`. Số ô giảm `3` (= `n`).

**Quan sát then chốt:** mỗi lệnh luôn đổi số ô một lượng **cố định**: `DOWN` `+n`, `UP` `−n`, `RIGHT` `+1`, `LEFT` `−1`. Vị trí cuối chỉ là **`0` cộng tất cả các độ lệch** đó.

Ví dụ 2: `0 + 3 + 1 − 3 = 1`. Đúng với đáp án!

In [8]:
n = 3
delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}
pos = 0
for c in ["DOWN", "RIGHT", "UP"]:
    pos += delta[c]
    print(f"{c:>5}: cộng {delta[c]:>2} -> ô {pos}")

 DOWN: cộng  3 -> ô 3
RIGHT: cộng  1 -> ô 4
   UP: cộng -3 -> ô 1


## 5. Cách 1 — Brute Force / baseline: dựng lưới thật rồi đi trên lưới

**Ý tưởng.** Làm đúng như đề: dựng lưới `grid` có `grid[i][j] = i * n + j`, đặt con rắn ở `(i, j) = (0, 0)`. Với mỗi lệnh, đổi `(i, j)` theo hướng; cuối cùng trả `grid[i][j]`.

Cách này **không sai**, chỉ **thừa**: dựng cả lưới tốn `O(n²)` bộ nhớ mà ta chỉ cần biết ô cuối, và công thức `i * n + j` ta có thể tính thẳng.

In [9]:
def final_position_brute(n: int, commands: List[str]) -> int:
    grid = [[i * n + j for j in range(n)] for i in range(n)]   # dựng lưới thật
    i = j = 0                                                  # rắn ở ô (0, 0) = ô số 0
    for c in commands:
        if c == "UP":
            i -= 1
        elif c == "DOWN":
            i += 1
        elif c == "LEFT":
            j -= 1
        else:                                                  # "RIGHT"
            j += 1
    return grid[i][j]


for n, commands, expected in examples:
    print(n, commands, "->", final_position_brute(n, commands), "(kỳ vọng", expected, ")")

2 ['RIGHT', 'DOWN'] -> 3 (kỳ vọng 3 )
3 ['DOWN', 'RIGHT', 'UP'] -> 1 (kỳ vọng 1 )


**Độ phức tạp.** Dựng lưới `O(n²)`, duyệt lệnh `O(m)` với `m = len(commands)`. Tổng `O(n² + m)`, bộ nhớ `O(n²)`.

**Nút thắt.** Lưới chỉ để tra cứu `grid[i][j] = i * n + j`, mà đó là một công thức tính được ngay. Bỏ lưới ⇒ `O(m)` time, `O(1)` space.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét liên tiếp:

1. **Không cần lưới.** Ô cuối là `i * n + j`, chỉ cần `i` và `j` ở cuối. Ta giữ hai biến `x` (hàng) và `y` (cột) và tính `x * n + y` **một lần ở cuối**.
2. **Không cần cả `x` và `y` riêng.** Mỗi lệnh đổi số ô một lượng cố định (`±1` hoặc `±n`), nên ta có thể giữ **một biến `pos`** và cộng thẳng độ lệch.

Từ nhận xét 2 suy thêm: phép cộng **giao hoán** (đổi thứ tự không đổi tổng) nên vị trí cuối **chỉ phụ thuộc số lượng** mỗi loại lệnh:

> `pos = (số DOWN − số UP) * n + (số RIGHT − số LEFT)`

(Thứ tự lệnh chỉ quan trọng để đảm bảo rắn không ra ngoài lưới — mà đề đã đảm bảo.)

In [10]:
n = 4
commands = ["RIGHT", "RIGHT", "DOWN", "DOWN", "LEFT", "UP", "DOWN"]
cnt = Counter(commands)
rows = cnt["DOWN"] - cnt["UP"]
cols = cnt["RIGHT"] - cnt["LEFT"]
print("DOWN - UP =", rows, "| RIGHT - LEFT =", cols)
print("ô cuối =", rows * n + cols)         # 2*4 + 1 = 9

DOWN - UP = 2 | RIGHT - LEFT = 1
ô cuối = 9


## 7. Các cách tiếp theo

Phần này có **4 cách** nữa (cùng Cách 1 là 5 cách). Mỗi cách: giải thích → code demo → code nộp LeetCode → độ phức tạp.

### Cách 2 — Theo dõi tọa độ `(x, y)`, tính `x * n + y` ở cuối

Giải thích từng đoạn:

- `x = y = 0`: hàng và cột hiện tại (rắn ở ô `0`).
- Với mỗi lệnh `c`, nhìn **ký tự đầu** `c[0]` để quyết định `x`/`y` tăng hay giảm: `U` giảm `x`, `D` tăng `x`, `L` giảm `y`, `R` tăng `y`.
- Cuối cùng `x * n + y` là số hiệu ô.

In [11]:
def final_position_xy(n: int, commands: List[str]) -> int:
    x = y = 0
    for c in commands:
        if c[0] == "U":
            x -= 1
        elif c[0] == "D":
            x += 1
        elif c[0] == "L":
            y -= 1
        else:                       # "R"
            y += 1
    return x * n + y


for n, commands, expected in examples:
    assert final_position_xy(n, commands) == expected
print("Cách 2 chạy đúng trên các ví dụ")

Cách 2 chạy đúng trên các ví dụ


In [12]:
# Code nộp LeetCode — Cách 2
class Solution:
    def finalPositionOfSnake(self, n: int, commands: List[str]) -> int:
        x = y = 0
        for c in commands:
            if c[0] == "U":
                x -= 1
            elif c[0] == "D":
                x += 1
            elif c[0] == "L":
                y -= 1
            else:
                y += 1
        return x * n + y


for n, commands, expected in examples:
    assert Solution().finalPositionOfSnake(n, commands) == expected
print("OK")

OK


**Độ phức tạp Cách 2:** Time `O(m)`, Space `O(1)`.

### Cách 3 — Giữ trực tiếp một biến `pos` với bảng độ lệch

Giải thích từng đoạn:

- `delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}`: nhớ quy tắc trong một bảng, không còn chuỗi `if / elif`.
- `pos = 0`, rồi `pos += delta[c]` cho mỗi lệnh.
- Trả `pos`. Không có phép nhân nào ở cuối, vì độ lệch đã chứa sẵn `n`.

In [13]:
def final_position_delta(n: int, commands: List[str]) -> int:
    delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}
    pos = 0
    for c in commands:
        pos += delta[c]
    return pos


for n, commands, expected in examples:
    assert final_position_delta(n, commands) == expected
print("Cách 3 chạy đúng trên các ví dụ")

Cách 3 chạy đúng trên các ví dụ


In [14]:
# Code nộp LeetCode — Cách 3
class Solution:
    def finalPositionOfSnake(self, n: int, commands: List[str]) -> int:
        delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}
        pos = 0
        for c in commands:
            pos += delta[c]
        return pos


for n, commands, expected in examples:
    assert Solution().finalPositionOfSnake(n, commands) == expected
print("OK")

OK


**Độ phức tạp Cách 3:** Time `O(m)`, Space `O(1)` (bảng `delta` có đúng `4` phần tử). Dạng `sum(delta[c] for c in commands)` cũng làm được trong một dòng.

### Cách 4 — Chỉ đếm số lệnh mỗi loại

Giải thích từng đoạn:

- `cnt = Counter(commands)` đếm cả bốn loại trong một lượt.
- Số hàng đã đi xuống (có thể âm) là `cnt["DOWN"] - cnt["UP"]`; số cột đã đi sang phải là `cnt["RIGHT"] - cnt["LEFT"]`.
- Đáp án `(DOWN − UP) * n + (RIGHT − LEFT)`.

Cách này nhấn mạnh tính **giao hoán**: đảo thứ tự các lệnh (miễn đề vẫn hợp lệ) không đổi đáp án.

In [15]:
def final_position_count(n: int, commands: List[str]) -> int:
    cnt = Counter(commands)
    return (cnt["DOWN"] - cnt["UP"]) * n + (cnt["RIGHT"] - cnt["LEFT"])


for n, commands, expected in examples:
    assert final_position_count(n, commands) == expected

# Phiên bản dùng list.count (không cần Counter)
def final_position_listcount(n: int, commands: List[str]) -> int:
    return (commands.count("DOWN") - commands.count("UP")) * n + commands.count("RIGHT") - commands.count("LEFT")


for n, commands, expected in examples:
    assert final_position_listcount(n, commands) == expected
print("Cách 4 chạy đúng trên các ví dụ")

Cách 4 chạy đúng trên các ví dụ


In [16]:
# Code nộp LeetCode — Cách 4
class Solution:
    def finalPositionOfSnake(self, n: int, commands: List[str]) -> int:
        cnt = Counter(commands)
        return (cnt["DOWN"] - cnt["UP"]) * n + (cnt["RIGHT"] - cnt["LEFT"])


for n, commands, expected in examples:
    assert Solution().finalPositionOfSnake(n, commands) == expected
print("OK")

OK


**Độ phức tạp Cách 4:** Time `O(m)`, Space `O(1)` (chỉ tối đa `4` khóa trong `Counter`).

### Cách 5 — `match` / `case` trên ký tự đầu (kiểu bài giải chính thức)

Giải thích từng đoạn: giống Cách 2 nhưng viết bằng `match c[0]:` với bốn `case`. Chạy được trên Python `3.10+`. Ưu điểm: đọc như bảng quy tắc; nhược điểm: phụ thuộc phiên bản Python của nơi chấm.

In [17]:
def final_position_match(n: int, commands: List[str]) -> int:
    x = y = 0
    for c in commands:
        match c[0]:
            case "U":
                x -= 1
            case "D":
                x += 1
            case "L":
                y -= 1
            case "R":
                y += 1
    return x * n + y


for n, commands, expected in examples:
    assert final_position_match(n, commands) == expected
print("Cách 5 chạy đúng trên các ví dụ")

Cách 5 chạy đúng trên các ví dụ


In [18]:
# Code nộp LeetCode — Cách 5
class Solution:
    def finalPositionOfSnake(self, n: int, commands: List[str]) -> int:
        x = y = 0
        for c in commands:
            match c[0]:
                case "U":
                    x -= 1
                case "D":
                    x += 1
                case "L":
                    y -= 1
                case "R":
                    y += 1
        return x * n + y


for n, commands, expected in examples:
    assert Solution().finalPositionOfSnake(n, commands) == expected
print("OK")

OK


**Độ phức tạp Cách 5:** Time `O(m)`, Space `O(1)`.

## 8. TRACE từng bước

Ta trace hai cách: Cách 2 (theo dõi `(x, y)`) trên ví dụ tự dựng `n = 4`, `commands = ["RIGHT", "RIGHT", "DOWN", "DOWN", "LEFT", "UP", "DOWN"]` (tự dựng; đáp án tính bằng code ngay bên dưới), và Cách 3 (biến `pos`) trên Ví dụ 2.

In [19]:
def trace_xy(n: int, commands: List[str]) -> int:
    print(f"n = {n}, commands = {commands}")
    x = y = 0
    print(f"  bắt đầu: x={x}, y={y}, ô = {x * n + y}")
    for step, c in enumerate(commands, start=1):
        if c[0] == "U":
            x -= 1
        elif c[0] == "D":
            x += 1
        elif c[0] == "L":
            y -= 1
        else:
            y += 1
        print(f"  lệnh {step}: {c:>5} -> x={x}, y={y}, ô = {x} * {n} + {y} = {x * n + y}")
    return x * n + y


assert trace_xy(4, ["RIGHT", "RIGHT", "DOWN", "DOWN", "LEFT", "UP", "DOWN"]) == 9

n = 4, commands = ['RIGHT', 'RIGHT', 'DOWN', 'DOWN', 'LEFT', 'UP', 'DOWN']
  bắt đầu: x=0, y=0, ô = 0
  lệnh 1: RIGHT -> x=0, y=1, ô = 0 * 4 + 1 = 1
  lệnh 2: RIGHT -> x=0, y=2, ô = 0 * 4 + 2 = 2
  lệnh 3:  DOWN -> x=1, y=2, ô = 1 * 4 + 2 = 6
  lệnh 4:  DOWN -> x=2, y=2, ô = 2 * 4 + 2 = 10
  lệnh 5:  LEFT -> x=2, y=1, ô = 2 * 4 + 1 = 9
  lệnh 6:    UP -> x=1, y=1, ô = 1 * 4 + 1 = 5
  lệnh 7:  DOWN -> x=2, y=1, ô = 2 * 4 + 1 = 9


In [20]:
def trace_delta(n: int, commands: List[str]) -> int:
    print(f"n = {n}, commands = {commands}")
    delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}
    pos = 0
    print("  bắt đầu: pos = 0")
    for step, c in enumerate(commands, start=1):
        before = pos
        pos += delta[c]
        print(f"  lệnh {step}: {c:>5} (độ lệch {delta[c]:>2}): {before} -> {pos}")
    return pos


assert trace_delta(3, ["DOWN", "RIGHT", "UP"]) == 1
assert trace_delta(2, ["RIGHT", "DOWN"]) == 3

n = 3, commands = ['DOWN', 'RIGHT', 'UP']
  bắt đầu: pos = 0
  lệnh 1:  DOWN (độ lệch  3): 0 -> 3
  lệnh 2: RIGHT (độ lệch  1): 3 -> 4
  lệnh 3:    UP (độ lệch -3): 4 -> 1
n = 2, commands = ['RIGHT', 'DOWN']
  bắt đầu: pos = 0
  lệnh 1: RIGHT (độ lệch  1): 0 -> 1
  lệnh 2:  DOWN (độ lệch  2): 1 -> 3


Bảng trace Cách 2 với ví dụ tự dựng `n = 4`:

| lệnh | `x` | `y` | ô = `x*4 + y` |
|---|---|---|---|
| (bắt đầu) | 0 | 0 | 0 |
| RIGHT | 0 | 1 | 1 |
| RIGHT | 0 | 2 | 2 |
| DOWN | 1 | 2 | 6 |
| DOWN | 2 | 2 | 10 |
| LEFT | 2 | 1 | 9 |
| UP | 1 | 1 | 5 |
| DOWN | 2 | 1 | 9 |

Cột `ô` đổi đúng `±4` khi đi dọc và `±1` khi đi ngang — chính là bảng `delta` của Cách 3.

## 9. Code nộp LeetCode

Bản nộp chọn **Cách 3** (bảng độ lệch): ngắn, không nhánh `if`, không phụ thuộc phiên bản Python.

In [21]:
class Solution:
    def finalPositionOfSnake(self, n: int, commands: List[str]) -> int:
        delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}   # mỗi lệnh đổi số ô một lượng cố định
        pos = 0                                                   # rắn bắt đầu ở ô 0
        for c in commands:
            pos += delta[c]
        return pos


sol = Solution()
for n, commands, expected in examples:
    print(n, commands, "->", sol.finalPositionOfSnake(n, commands), "(kỳ vọng", expected, ")")

2 ['RIGHT', 'DOWN'] -> 3 (kỳ vọng 3 )
3 ['DOWN', 'RIGHT', 'UP'] -> 1 (kỳ vọng 1 )


## 10. Vì sao thuật toán đúng

- **Công thức ô.** Ô `(i, j)` mang số `i * n + j`. Khi đi `DOWN`, `i` tăng `1` nên số ô tăng đúng `n`; `UP` giảm `n`; `RIGHT` tăng `j` nên số ô tăng `1`; `LEFT` giảm `1`. Vậy mỗi lệnh cộng một hằng số vào số ô.
- **Không cần kiểm tra biên.** Đề đảm bảo rắn không ra ngoài lưới, nên mọi `(i, j)` trung gian đều hợp lệ và công thức `i * n + j` luôn đúng. (Nếu rắn có thể "tràn" khỏi cạnh phải, ô `i * n + n` trùng với ô `(i + 1) * n + 0` — đó là lý do đề phải đảm bảo điều kiện này.)
- **Quy nạp.** Trước lệnh đầu `pos = 0` đúng (ô `(0, 0)`). Nếu sau `k` lệnh `pos` đúng, thì sau lệnh thứ `k + 1` `pos` cũng đúng nhờ quy tắc trên.
- **Giao hoán (Cách 4).** `pos` cuối là tổng các độ lệch, phép cộng không phụ thuộc thứ tự ⇒ chỉ cần số lượng từng lệnh.
- **Đối chiếu độc lập.** Cách 1 (dựng lưới thật) được so với các cách còn lại trên hàng nghìn bộ lệnh ngẫu nhiên hợp lệ ở mục 12.

## 11. Time / Space Complexity

Gọi `m = len(commands)`.

| Cách | Time | Space |
|---|---|---|
| 1. Dựng lưới thật | `O(n² + m)` | `O(n²)` |
| 2. Tọa độ `(x, y)` | `O(m)` | `O(1)` |
| 3. Biến `pos` + bảng `delta` | `O(m)` | `O(1)` |
| 4. Đếm số lệnh | `O(m)` | `O(1)` |
| 5. `match` / `case` | `O(m)` | `O(1)` |

Mọi cách đều phải **đọc hết** `commands` nên `O(m)` là tối ưu về mặt tiệm cận. Với `m <= 100` mọi cách đều tức thì.

## 12. Bộ test

Gồm: ví dụ đề, biên (một lệnh, lưới nhỏ nhất `n = 2`, đi một vòng quay về ô `0`, đi tới góc dưới phải), đối chiếu các cách với nhau, và ngẫu nhiên có seed — các lệnh được sinh sao cho rắn **luôn ở trong lưới** (đúng ràng buộc đề).

In [22]:
IMPLS = {
    "brute": final_position_brute,
    "xy": final_position_xy,
    "delta": final_position_delta,
    "count": final_position_count,
    "listcount": final_position_listcount,
    "match": final_position_match,
    "Solution": Solution().finalPositionOfSnake,
}

# 1) ví dụ đề
for n, commands, expected in examples:
    for name, f in IMPLS.items():
        assert f(n, commands) == expected, (name, n, commands)

# 2) biên và các trường hợp tự dựng (đáp án tính tay)
manual = [
    (2, ["RIGHT"], 1),                                  # một lệnh
    (2, ["DOWN"], 2),                                   # một lệnh xuống
    (2, ["RIGHT", "DOWN", "LEFT", "UP"], 0),            # đi một vòng quay về ô 0
    (3, ["DOWN", "DOWN", "RIGHT", "RIGHT"], 8),         # tới góc dưới phải của lưới 3x3
    (10, ["DOWN"] * 9 + ["RIGHT"] * 9, 99),             # góc dưới phải lưới 10x10
    (4, ["RIGHT", "DOWN", "UP", "LEFT"], 0),            # đi rồi quay lại ngay
]
for n, commands, expected in manual:
    for name, f in IMPLS.items():
        assert f(n, commands) == expected, (name, n, commands, f(n, commands), expected)
print("ví dụ đề + biên: PASS")

ví dụ đề + biên: PASS


In [23]:
# 3) ngẫu nhiên có seed: sinh bước đi luôn hợp lệ (rắn không rời lưới)
def random_case(rng):
    n = rng.randint(2, 10)
    m = rng.randint(1, 100)
    x = y = 0
    commands = []
    for _ in range(m):
        options = []
        if x > 0:
            options.append("UP")
        if x < n - 1:
            options.append("DOWN")
        if y > 0:
            options.append("LEFT")
        if y < n - 1:
            options.append("RIGHT")
        c = rng.choice(options)
        commands.append(c)
        if c == "UP":
            x -= 1
        elif c == "DOWN":
            x += 1
        elif c == "LEFT":
            y -= 1
        else:
            y += 1
    return n, commands, x * n + y          # đáp án đúng theo định nghĩa (tọa độ cuối)


rng = random.Random(3248)
for _ in range(3000):
    n, commands, expected = random_case(rng)
    for name, f in IMPLS.items():
        assert f(n, commands) == expected, (name, n, commands)
print("3000 test ngẫu nhiên: tất cả các cách đồng nhất — PASS")

3000 test ngẫu nhiên: tất cả các cách đồng nhất — PASS


In [24]:
# 4) tính chất giao hoán: xáo trộn lệnh (nếu vẫn hợp lệ) không đổi đáp án; ở đây chỉ kiểm tra công thức đếm
rng = random.Random(7)
for _ in range(500):
    n, commands, expected = random_case(rng)
    shuffled = commands[:]
    rng.shuffle(shuffled)
    # công thức đếm không phụ thuộc thứ tự, nên cho cùng kết quả dù thứ tự bị xáo
    assert final_position_count(n, shuffled) == expected
print("giao hoán: PASS")

giao hoán: PASS


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đảo chiều `UP` / `DOWN`

Dòng đầu của lưới là hàng `0`, nên **đi lên là hàng giảm** (`x -= 1`), không phải tăng. Trong toán học ta hay quen trục `y` hướng lên, nhưng với lưới chỉ số thì hàng tăng xuống dưới.

In [25]:
def sai_dao_chieu(n, commands):
    x = y = 0
    for c in commands:
        if c[0] == "U":
            x += 1            # SAI: lên mà lại tăng hàng
        elif c[0] == "D":
            x -= 1            # SAI
        elif c[0] == "L":
            y -= 1
        else:
            y += 1
    return x * n + y

print("sai:", sai_dao_chieu(3, ["DOWN", "RIGHT", "UP"]), "(đúng là 1)")   # ra 1 do hai lệnh đối xứng triệt tiêu -> dễ lọt!
print("sai:", sai_dao_chieu(2, ["RIGHT", "DOWN"]), "(đúng là 3)")        # lộ ra: -2 + 1 = -1


def dung_dao_chieu(n, commands):
    x = y = 0
    for c in commands:
        if c[0] == "U":
            x -= 1
        elif c[0] == "D":
            x += 1
        elif c[0] == "L":
            y -= 1
        else:
            y += 1
    return x * n + y

print("đúng:", dung_dao_chieu(3, ["DOWN", "RIGHT", "UP"]), dung_dao_chieu(2, ["RIGHT", "DOWN"]))

sai: 1 (đúng là 1)
sai: -1 (đúng là 3)
đúng: 1 3


Nhận xét: ví dụ 2 vẫn ra `1` dù code sai, vì `DOWN` và `UP` bị đổi cho nhau nên tổng không đổi. Đó là lý do **không được chỉ test một ví dụ** — ví dụ 1 mới bắt được lỗi.

### Lỗi 2: So sánh ký tự đơn khi lệnh là cả chuỗi

`c == "U"` luôn `False` vì `c` là `"UP"` (hai ký tự). Code không báo lỗi, chỉ trả sai (thường là `0`).

In [26]:
def sai_so_sanh(n, commands):
    pos = 0
    for c in commands:
        if c == "U":              # SAI: c là "UP"
            pos -= n
        elif c == "D":
            pos += n
        elif c == "L":
            pos -= 1
        elif c == "R":
            pos += 1
    return pos

print("sai:", sai_so_sanh(2, ["RIGHT", "DOWN"]), "(đúng là 3)")   # 0, không lệnh nào khớp


def dung_so_sanh(n, commands):
    pos = 0
    for c in commands:
        if c == "UP":             # đúng: so với cả chuỗi, hoặc dùng c[0] == "U"
            pos -= n
        elif c == "DOWN":
            pos += n
        elif c == "LEFT":
            pos -= 1
        elif c == "RIGHT":
            pos += 1
    return pos

print("đúng:", dung_so_sanh(2, ["RIGHT", "DOWN"]))

sai: 0 (đúng là 3)
đúng: 3


### Lỗi 3: Nhầm công thức `i * n + j` thành `i + j * n`

Đề đánh số theo **hàng** (đọc ngang trước). Nếu viết `x + y * n` thì bạn đang đánh số theo **cột**, đáp án sai khi `x != y` .

In [27]:
def sai_cong_thuc(n, commands):
    x = y = 0
    for c in commands:
        if c[0] == "U":
            x -= 1
        elif c[0] == "D":
            x += 1
        elif c[0] == "L":
            y -= 1
        else:
            y += 1
    return x + y * n              # SAI: đánh số theo cột

print("sai:", sai_cong_thuc(2, ["RIGHT", "DOWN"]), "(đúng là 3)")        # 1 + 1*2 = 3 -> trùng ngẫu nhiên!
print("sai:", sai_cong_thuc(3, ["DOWN", "DOWN", "RIGHT"]), "(đúng là 7)")  # 2 + 1*3 = 5, sai
print("đúng:", final_position_xy(3, ["DOWN", "DOWN", "RIGHT"]))

sai: 3 (đúng là 3)
sai: 5 (đúng là 7)
đúng: 7


### Lỗi 4: Dùng `n` sai khi nhảy một hàng (hoặc quên nhân `n`)

Đi `DOWN` là sang **hàng kế**, cách `n` ô chứ không phải `1` ô. Lỗi hay gặp là viết `pos += 1` cho cả `DOWN` và `RIGHT`.

In [28]:
def sai_quen_n(n, commands):
    delta = {"UP": -1, "DOWN": 1, "LEFT": -1, "RIGHT": 1}     # SAI: UP/DOWN phải là ∓n
    pos = 0
    for c in commands:
        pos += delta[c]
    return pos

print("sai:", sai_quen_n(2, ["RIGHT", "DOWN"]), "(đúng là 3)")     # 2

def dung_n(n, commands):
    delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}
    return sum(delta[c] for c in commands)

print("đúng:", dung_n(2, ["RIGHT", "DOWN"]))

sai: 2 (đúng là 3)
đúng: 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Dấu hiệu "mô phỏng vị trí":** "bắt đầu từ ..., làm theo các lệnh ..., trả về vị trí cuối". Mỗi lệnh là một **bước đổi trạng thái cố định** ⇒ giữ biến trạng thái và cộng dồn.
- **Lưới phẳng hóa:** ô `(i, j)` ⇄ số `i * n + j`; ngược lại `(p // n, p % n)`. Lên/xuống = `∓n`, trái/phải = `∓1`. Mẹo này dùng rất nhiều trong bài lưới (flatten grid).
- **Bảng hướng:** gom bốn hướng vào `dict` hoặc list `dirs` (`(−1, 0), (1, 0), (0, −1), (0, 1)`); chuẩn bị tốt cho bài BFS/DFS trên lưới.
- **Khi các bước chỉ cộng:** nếu tác động của lệnh không phụ thuộc trạng thái hiện tại (không có tường, không có chướng ngại) thì thứ tự không quan trọng, chỉ cần đếm.
- **Test ví dụ đối xứng cẩn thận:** như Lỗi 1, một ví dụ có `UP` và `DOWN` triệt tiêu có thể che lỗi đảo hướng. Hãy tự thêm ví dụ lệch.
- **Đọc kỹ đề:** lệnh là `"UP"` (chuỗi cả chữ), và đề đã **đảm bảo** không ra ngoài lưới, nên không cần `if` kiểm tra biên.

## 15. Biến thể của bài

1. **Có thể ra ngoài lưới.** Nếu một lệnh đưa rắn ra ngoài thì **bỏ qua lệnh** đó. Khi đó phải giữ `(x, y)` và kiểm tra biên trước khi đi — không thể chỉ đếm số lệnh.
2. **Số ô đã đi qua.** Đếm số ô **khác nhau** rắn từng đứng: lưu các `pos` trong một `set`.
3. **Trả về đường đi.** Trả về danh sách mọi ô rắn đi qua (kể cả ô `0`).
4. **Bắt đầu ở ô bất kỳ `s`.** Chỉ cần đặt `pos = s` thay vì `0` (Cách 3), hoặc đổi `s` thành `(s // n, s % n)` rồi tiếp tục như Cách 2.
5. **Lưới `R x C`.** Độ rộng hàng là `C` chứ không phải `n`: ô `(i, j)` mang số `i * C + j`; `UP` / `DOWN` đổi `∓C`.

In [29]:
# Minh họa biến thể 1 (bỏ qua lệnh ra ngoài) và biến thể 2 (số ô khác nhau đã đi qua)
def final_position_clamped(n: int, commands: List[str]) -> int:
    x = y = 0
    for c in commands:
        nx, ny = x, y
        if c == "UP":
            nx -= 1
        elif c == "DOWN":
            nx += 1
        elif c == "LEFT":
            ny -= 1
        else:
            ny += 1
        if 0 <= nx < n and 0 <= ny < n:       # chỉ đi khi còn trong lưới
            x, y = nx, ny
    return x * n + y


def distinct_cells_visited(n: int, commands: List[str]) -> int:
    delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}
    pos = 0
    seen = {0}
    for c in commands:
        pos += delta[c]
        seen.add(pos)
    return len(seen)


print(final_position_clamped(2, ["UP", "LEFT", "RIGHT", "DOWN", "DOWN"]))    # bị chặn ở biên, ra 3
print(distinct_cells_visited(3, ["DOWN", "RIGHT", "UP", "LEFT", "DOWN"]))    # 0,3,4,1,0,3 -> 4 ô khác nhau

3
4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng lưới | Làm đúng như đề, tra `grid[i][j]` | `O(n² + m)` | `O(n²)` | Để hiểu đề; kiểm chứng (đối chiếu) |
| 2. `(x, y)` | Giữ hàng/cột, tính `x*n+y` cuối | `O(m)` | `O(1)` | Khi cần biết cả hàng và cột (lưới, biên) |
| 3. `pos` + `delta` | Cộng độ lệch cố định vào số ô | `O(m)` | `O(1)` | Cách gọn nhất để nộp |
| 4. Đếm lệnh | `(D−U)*n + (R−L)` | `O(m)` | `O(1)` | Khi thứ tự không quan trọng; nhìn ra bản chất |
| 5. `match` / `case` | Như Cách 2, cú pháp `match` | `O(m)` | `O(1)` | Python 3.10+, thích code đọc như bảng quy tắc |

Lời khuyên: dùng **Cách 3** để nộp; nhớ **Cách 2** làm nền cho các bài lưới có tường/biên (BFS, spiral matrix...).

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `divmod(7, 3)`, `Solution().finalPositionOfSnake(3, ["DOWN", "DOWN"])`, `Solution().finalPositionOfSnake(4, ["RIGHT", "RIGHT", "DOWN"])`, `Counter(["UP", "UP", "LEFT"])["UP"]`.

**Bài 2 (tính tay).** Với `n = 4`, `commands = ["DOWN", "RIGHT", "RIGHT", "DOWN", "LEFT"]`: vẽ lưới `4 x 4` đánh số `0..15`, đi từng bước và cho biết ô cuối và `(hàng, cột)` của nó.

**Bài 3 (điền TODO).** Hoàn thành `final_position_todo(n, commands)` theo Cách 3 (biến `pos` + bảng `delta`).

In [30]:
def final_position_todo(n, commands):
    # TODO 1: tạo bảng delta = {"UP": ..., "DOWN": ..., "LEFT": ..., "RIGHT": ...} (dùng n cho UP/DOWN)
    # TODO 2: pos = 0, rồi duyệt từng lệnh c trong commands và cộng delta[c] vào pos
    # TODO 3: trả về pos
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn lời giải, viết `snake_cells_visited(n, commands)` trả về **danh sách** các ô rắn đi qua theo thứ tự (kể cả ô `0` lúc đầu và các ô lặp lại). Ví dụ `n = 2`, `["RIGHT", "DOWN"]` → `[0, 1, 3]`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `snake_start_at(n, start, commands)` với rắn bắt đầu ở ô `start` thay vì `0` (đề vẫn đảm bảo không ra ngoài). (b) Viết `snake_rc(n, commands)` trả về cặp `(hàng, cột)` của ô cuối. (c) Cho `n` và **ô cuối `p`**, viết `shortest_commands(n, p)` trả về một danh sách lệnh **ngắn nhất** đưa rắn từ ô `0` tới `p` (hàng đi xuống `DOWN`, cột đi `RIGHT`; chỉ cần một đáp án hợp lệ nên kiểm tra bằng hàm `Solution`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `divmod(7, 3) = (2, 1)`. `DOWN` làm số ô tăng `n`; `RIGHT` tăng `1`; `Counter` đếm số lần xuất hiện. Bài 2: vẽ lưới ra giấy, mỗi bước viết `(hàng, cột)`.
- **Mức 2 (vừa):** Bài 3: `delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}`. Bài 4: tạo list `[0]` rồi `append(pos)` sau mỗi lệnh. Bài 5(a): chỉ đổi `pos = 0` thành `pos = start`.
- **Mức 3 (sâu):** Bài 5(b): sau khi có ô cuối `p`, dùng `divmod(p, n)`. Bài 5(c): `row, col = divmod(p, n)` rồi trả `["DOWN"] * row + ["RIGHT"] * col`; vì `p < n²` nên `col < n` và rắn không ra ngoài.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert divmod(7, 3) == (2, 1)
assert Solution().finalPositionOfSnake(3, ["DOWN", "DOWN"]) == 6          # 2 hàng xuống: 2 * 3
assert Solution().finalPositionOfSnake(4, ["RIGHT", "RIGHT", "DOWN"]) == 6   # 1*4 + 2
assert Counter(["UP", "UP", "LEFT"])["UP"] == 2

# Bài 2: n = 4, DOWN -> 4, RIGHT -> 5, RIGHT -> 6, DOWN -> 10, LEFT -> 9
n, commands = 4, ["DOWN", "RIGHT", "RIGHT", "DOWN", "LEFT"]
assert Solution().finalPositionOfSnake(n, commands) == 9
assert divmod(9, n) == (2, 1)


# Bài 3
def final_position_todo_ans(n, commands):
    delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}
    pos = 0
    for c in commands:
        pos += delta[c]
    return pos


for n, commands, expected in examples:
    assert final_position_todo_ans(n, commands) == expected


# Bài 4
def snake_cells_visited(n, commands):
    delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}
    pos = 0
    path = [0]
    for c in commands:
        pos += delta[c]
        path.append(pos)
    return path


assert snake_cells_visited(2, ["RIGHT", "DOWN"]) == [0, 1, 3]
assert snake_cells_visited(3, ["DOWN", "RIGHT", "UP"]) == [0, 3, 4, 1]


# Bài 5
def snake_start_at(n, start, commands):
    delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}
    pos = start
    for c in commands:
        pos += delta[c]
    return pos


def snake_rc(n, commands):
    return divmod(Solution().finalPositionOfSnake(n, commands), n)


def shortest_commands(n, p):
    row, col = divmod(p, n)
    return ["DOWN"] * row + ["RIGHT"] * col


assert snake_start_at(3, 4, ["UP", "LEFT"]) == 0               # từ ô 4 (tâm lưới 3x3): lên ô 1, trái ô 0
assert snake_start_at(3, 4, ["DOWN", "RIGHT"]) == 8
assert snake_rc(3, ["DOWN", "RIGHT", "UP"]) == (0, 1)
assert snake_rc(4, ["DOWN", "RIGHT", "RIGHT", "DOWN", "LEFT"]) == (2, 1)
for n in range(2, 8):
    for p in range(n * n):
        cmds = shortest_commands(n, p)
        assert Solution().finalPositionOfSnake(n, cmds) == p
        assert len(cmds) == p // n + p % n                      # số lệnh = hàng + cột (ngắn nhất)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mỗi lệnh cộng một độ lệch cố định; số ô = `hàng * n + cột`)

```python
class Solution:
    def finalPositionOfSnake(self, n: int, commands: List[str]) -> int:
        delta = {"UP": -n, "DOWN": n, "LEFT": -1, "RIGHT": 1}
        pos = 0
        for c in commands:
            pos += delta[c]
        return pos
```

**Checklist**

- [ ] `UP` / `DOWN` đổi `∓n`, `LEFT` / `RIGHT` đổi `∓1` (không đảo dấu)?
- [ ] So sánh với **chuỗi đầy đủ** `"UP"` (hoặc ký tự đầu `c[0] == "U"`), không so `c == "U"`?
- [ ] Công thức ô là `hàng * n + cột` (đánh số theo hàng)?
- [ ] Rắn bắt đầu ở ô `0` (`pos = 0`)?
- [ ] Đã thử cả hai ví dụ đề (`3` và `1`) và một ví dụ **không đối xứng** để bắt lỗi đảo hướng?
- [ ] Skeleton đúng `class Solution` / `def finalPositionOfSnake(self, n: int, commands: List[str]) -> int`?